# StepByStepReasoner: Model Comparison + Investment Parameterization

**Experiment**: Does the StepByStepReasoner template make lightweight models reason
as well as expensive reasoning models? And can investment-level parameterization
(quick/standard/thorough) control depth without degrading correctness?

This is a harder test than DataAnalyzer — we measure **correctness** (did the model
get the right answer?), not just section coverage.

**Matrix**: 3 models x 3 investments x 3 problems = 27 runs

**Requires**: `OPENAI_API_KEY`. Set `EXECUTE_LLM = True` to call the API.

In [3]:
%load_ext autoreload
%autoreload 2

import os
import json
import re

import pandas as pd

# Set your API key here or via environment variable
os.environ.setdefault('OPENAI_API_KEY', 'sk-...')  # Set your key here or via environment variable


EXECUTE_LLM = True

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Setup: Models, Problems, and Investment Levels

In [4]:
from mycontext.templates.free.reasoning import StepByStepReasoner
from mycontext import Context
from mycontext.foundation import Directive

reasoner = StepByStepReasoner()

MODELS = {
    "gpt-4o-mini": {"label": "lightweight", "reasoning": False},
    "gpt-4o":      {"label": "mid-tier",    "reasoning": False},
    "gpt-5.2":     {"label": "reasoning",   "reasoning": True},
}

PROBLEMS = [
    {
        "id": "avg_speed",
        "label": "Simple math",
        "problem": (
            "A train travels 120km in 2 hours, then speeds up to travel "
            "200km in the next 2.5 hours. What is the average speed for "
            "the entire journey?"
        ),
        "domain": "mathematical",
        "correct_answer": 71.11,
        "unit": "km/h",
        "tolerance": 0.5,
    },
    {
        "id": "discount",
        "label": "Multi-step logic",
        "problem": (
            "A store offers 20% off, then an additional 15% off the "
            "discounted price. Is this the same as 35% off? What is the "
            "actual total discount percentage?"
        ),
        "domain": "mathematical",
        "correct_answer": 32.0,
        "unit": "%",
        "tolerance": 0.5,
    },
    {
        "id": "uptime",
        "label": "Applied reasoning",
        "problem": (
            "A company has 3 servers. Each has 99.5% uptime independently. "
            "If ALL 3 must be running for the service to work, what is the "
            "overall uptime percentage and the expected downtime in hours per year?"
        ),
        "domain": "mathematical",
        "correct_answer": 98.51,
        "unit": "% uptime",
        "tolerance": 0.1,
    },
]

INVESTMENT_CONSTRAINTS = {
    "quick": (
        " Constraint: Be concise. Use brief bullet points for each step. "
        "Skip lengthy elaboration. Maximum 3 sub-steps in the EXECUTE phase."
    ),
    "standard": "",
    "thorough": (
        " Constraint: Be extremely thorough. Show detailed sub-steps, "
        "multiple verification methods, and comprehensive explanations. "
        "Show at least 2 alternative verification approaches."
    ),
}

STEPS = ["UNDERSTAND", "PLAN", "EXECUTE", "VERIFY", "CONCLUDE"]

print(f"Models:      {list(MODELS.keys())}")
print(f"Problems:    {[p['id'] for p in PROBLEMS]}")
print(f"Investments: {list(INVESTMENT_CONSTRAINTS.keys())}")
print(f"Total runs:  {len(MODELS) * len(PROBLEMS) * len(INVESTMENT_CONSTRAINTS)}")

Models:      ['gpt-4o-mini', 'gpt-4o', 'gpt-5.2']
Problems:    ['avg_speed', 'discount', 'uptime']
Investments: ['quick', 'standard', 'thorough']
Total runs:  27


## 2. Run Function

In [5]:
def _extract(result) -> str:
    if hasattr(result, "response"):
        return result.response or ""
    return str(result)


def count_steps(text: str) -> dict:
    """Check which of the 5 reasoning steps are present."""
    upper = text.upper()
    return {step: step in upper for step in STEPS}


def extract_number(text: str) -> float | None:
    """Extract the final numeric answer from the CONCLUDE section.

    Multi-strategy: answer-keyword patterns first, then unit-tagged numbers
    (last match), then fallback to last number in range. Fixes mis-extraction
    on verbose models that restate intermediate values before the answer.
    """
    upper = text.upper()
    conclude_idx = upper.find("CONCLUDE")
    search_text = text[conclude_idx:] if conclude_idx != -1 else text[-1500:]

    def _valid(s):
        try:
            v = float(s.replace(",", ""))
            return v if 0.01 < v < 100000 else None
        except ValueError:
            return None

    answer_pats = [
        r"(?:the\s+)?(?:answer|result|total\s+discount|actual\s+discount|overall\s+uptime|average\s+speed|combined\s+uptime)\s*(?:is|=|:)\s*(?:approximately\s*|about\s*|~\s*)?(\d[\d,]*\.?\d*)",
        r"(?:therefore|thus|hence|so)[,:]?\s+(?:the\s+)?(?:\w+\s+){0,4}(?:is|=|:)\s*(?:approximately\s*|about\s*|~\s*)?(\d[\d,]*\.?\d*)",
        r"\*\*(\d[\d,]*\.?\d*)\s*(?:%|km/h)\*\*",
    ]
    for pat in answer_pats:
        for m in re.finditer(pat, search_text, re.IGNORECASE):
            v = _valid(m.group(1))
            if v is not None:
                return v

    unit_matches = list(re.finditer(
        r"(\d[\d,]*\.?\d*)\s*(?:%|km/h|hours?\s+per\s+year)", search_text
    ))
    if unit_matches:
        v = _valid(unit_matches[-1].group(1))
        if v is not None:
            return v

    all_numbers = re.findall(r"\d[\d,]*\.?\d*", search_text)
    candidates = [_valid(n) for n in all_numbers]
    candidates = [c for c in candidates if c is not None]
    return candidates[-1] if candidates else None


def run_reasoner(model: str, problem: dict, investment: str) -> dict:
    """Run a single (model x problem x investment) combination."""
    prob_text = problem["problem"]
    constraint = INVESTMENT_CONSTRAINTS[investment]
    if constraint:
        prob_text = prob_text + constraint

    if EXECUTE_LLM:
        result = reasoner.execute(
            provider="openai",
            model=model,
            problem=prob_text,
            domain=problem["domain"],
            use_cache=False,
        )
        out = _extract(result)
        meta = result.metadata if hasattr(result, "metadata") else {}
        tokens = result.tokens_used if hasattr(result, "tokens_used") else 0
        cost = result.cost_usd if hasattr(result, "cost_usd") else 0
    else:
        out = "(execute=False)"
        meta, tokens, cost = {}, 0, 0

    steps_found = count_steps(out)
    extracted_answer = extract_number(out)
    correct_answer = problem["correct_answer"]
    tolerance = problem["tolerance"]

    if extracted_answer is not None:
        error = abs(extracted_answer - correct_answer)
        is_correct = error <= tolerance
        pct_error = (error / correct_answer) * 100 if correct_answer else 0
    else:
        error, is_correct, pct_error = None, False, None

    return {
        "model": model,
        "model_type": MODELS[model]["label"],
        "problem_id": problem["id"],
        "problem_label": problem["label"],
        "investment": investment,
        "output_chars": len(out),
        "steps_found": sum(steps_found.values()),
        "steps_detail": steps_found,
        "extracted_answer": extracted_answer,
        "correct_answer": correct_answer,
        "is_correct": is_correct,
        "pct_error": pct_error,
        "input_tokens": meta.get("input_tokens", 0),
        "output_tokens": meta.get("output_tokens", 0),
        "total_tokens": tokens,
        "cost_usd": cost,
        "finish_reason": meta.get("finish_reason", ""),
        "full_output": out,
    }


print("Run function ready.")

Run function ready.


## 3. Execute All 27 Runs

In [6]:
all_results = []
total = len(MODELS) * len(PROBLEMS) * len(INVESTMENT_CONSTRAINTS)
i = 0

for model_name in MODELS:
    for problem in PROBLEMS:
        for investment in INVESTMENT_CONSTRAINTS:
            i += 1
            tag = f"{model_name} x {problem['id']} x {investment}"
            print(f"{i:2d}/{total}  {tag}...", end=" ", flush=True)
            try:
                r = run_reasoner(model_name, problem, investment)
                all_results.append(r)
                mark = "correct" if r["is_correct"] else "WRONG"
                print(
                    f"ans={r['extracted_answer']}  [{mark}]  "
                    f"{r['output_chars']} chars  {r['total_tokens']} tok  "
                    f"${r['cost_usd']:.4f}"
                )
            except Exception as e:
                print(f"ERROR: {e}")
                all_results.append({
                    "model": model_name,
                    "model_type": MODELS[model_name]["label"],
                    "problem_id": problem["id"],
                    "problem_label": problem["label"],
                    "investment": investment,
                    "output_chars": 0, "steps_found": 0,
                    "steps_detail": {}, "extracted_answer": None,
                    "correct_answer": problem["correct_answer"],
                    "is_correct": False, "pct_error": None,
                    "input_tokens": 0, "output_tokens": 0,
                    "total_tokens": 0, "cost_usd": 0,
                    "finish_reason": f"ERROR: {e}", "full_output": "",
                })

print(f"\nDone - {len(all_results)} runs complete.")

 1/27  gpt-4o-mini x avg_speed x quick... ans=71.11  [correct]  3588 chars  1751 tok  $0.0007
 2/27  gpt-4o-mini x avg_speed x standard... ans=71.11  [correct]  4076 chars  1852 tok  $0.0007
 3/27  gpt-4o-mini x avg_speed x thorough... ans=71.11  [correct]  4463 chars  2057 tok  $0.0009
 4/27  gpt-4o-mini x discount x quick... ans=32.0  [correct]  3895 chars  1823 tok  $0.0007
 5/27  gpt-4o-mini x discount x standard... ans=32.0  [correct]  4618 chars  1949 tok  $0.0008
 6/27  gpt-4o-mini x discount x thorough... ans=20.0  [WRONG]  4755 chars  2110 tok  $0.0009
 7/27  gpt-4o-mini x uptime x quick... ans=98.51  [correct]  4068 chars  1870 tok  $0.0007
 8/27  gpt-4o-mini x uptime x standard... ans=98.51  [correct]  4454 chars  1883 tok  $0.0008
 9/27  gpt-4o-mini x uptime x thorough... ans=98.51  [correct]  4525 chars  1981 tok  $0.0008
10/27  gpt-4o x avg_speed x quick... ans=71.11  [correct]  3232 chars  1721 tok  $0.0109
11/27  gpt-4o x avg_speed x standard... ans=71.11  [correct]  32

In [ ]:
print("=" * 75)
print("EXTRACTION DIAGNOSTIC — gpt-5.2 CONCLUDE sections")
print("=" * 75)
for r in all_results:
    if r["model"] != "gpt-5.2":
        continue
    out = r.get("full_output", "")
    upper = out.upper()
    idx = upper.find("CONCLUDE")
    conclude_text = out[idx:idx+600] if idx != -1 else "(CONCLUDE not found)"
    old_ans = r["extracted_answer"]
    new_ans = extract_number(out)
    mark = "OK" if r["is_correct"] else "MISS"
    print(f"\n--- {r['problem_id']} / {r['investment']} ---")
    print(f"  Extracted: {old_ans}  Correct: {r['correct_answer']}  [{mark}]")
    print(f"  Re-extracted (new logic): {new_ans}")
    print(f"  CONCLUDE snippet:\n{conclude_text[:500]}")
print("\n" + "=" * 75)

## 4. Results Table

In [7]:
df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("full_output", "steps_detail")}
    for r in all_results
])

display_cols = [
    "model", "problem_id", "investment",
    "is_correct", "extracted_answer", "correct_answer", "pct_error",
    "steps_found", "output_chars", "total_tokens", "cost_usd",
]

df[display_cols].style.format({
    "cost_usd": "${:.4f}",
    "pct_error": "{:.1f}%",
    "extracted_answer": "{:.2f}",
    "correct_answer": "{:.2f}",
}).map(
    lambda v: "background-color: #c6efce" if v is True else
              ("background-color: #ffc7ce" if v is False else ""),
    subset=["is_correct"]
).background_gradient(
    subset=["cost_usd"], cmap="RdYlGn_r"
)

,model,problem_id,investment,is_correct,extracted_answer,correct_answer,pct_error,steps_found,output_chars,total_tokens,cost_usd
0,gpt-4o-mini,avg_speed,quick,True,71.11,71.11,0.0%,5,3588,1751,$0.0007
1,gpt-4o-mini,avg_speed,standard,True,71.11,71.11,0.0%,5,4076,1852,$0.0007
2,gpt-4o-mini,avg_speed,thorough,True,71.11,71.11,0.0%,5,4463,2057,$0.0009
3,gpt-4o-mini,discount,quick,True,32.00,32.00,0.0%,5,3895,1823,$0.0007
4,gpt-4o-mini,discount,standard,True,32.00,32.00,0.0%,5,4618,1949,$0.0008
5,gpt-4o-mini,discount,thorough,False,20.00,32.00,37.5%,5,4755,2110,$0.0009
6,gpt-4o-mini,uptime,quick,True,98.51,98.51,0.0%,5,4068,1870,$0.0007
7,gpt-4o-mini,uptime,standard,True,98.51,98.51,0.0%,5,4454,1883,$0.0008
8,gpt-4o-mini,uptime,thorough,True,98.51,98.51,0.0%,5,4525,1981,$0.0008
9,gpt-4o,avg_speed,quick,True,71.11,71.11,0.0%,5,3232,1721,$0.0109


## 5. Correctness Analysis

In [8]:
print("=" * 75)
print("CORRECTNESS: Did each model get the right answer?")
print("=" * 75)

for problem in PROBLEMS:
    pid = problem["id"]
    subset = df[df["problem_id"] == pid]
    print(f"\n--- {problem['label']} ({pid}) --- correct: {problem['correct_answer']} {problem['unit']}")
    print(f"  {'Model':<15s} {'Investment':<12s} {'Answer':>10s} {'Correct?':>10s} {'Error':>8s} {'Cost':>10s}")
    print(f"  {'-'*65}")
    for _, row in subset.iterrows():
        ans_str = f"{row['extracted_answer']:.2f}" if pd.notna(row['extracted_answer']) else "N/A"
        err_str = f"{row['pct_error']:.1f}%" if pd.notna(row['pct_error']) else "N/A"
        mark = "YES" if row['is_correct'] else "NO"
        print(
            f"  {row['model']:<15s} {row['investment']:<12s} "
            f"{ans_str:>10s} {mark:>10s} {err_str:>8s} ${row['cost_usd']:>9.4f}"
        )

# Accuracy summary by model
print("\n" + "=" * 75)
print("ACCURACY RATE BY MODEL (across all problems and investments)")
print("=" * 75)
for model_name in MODELS:
    subset = df[df["model"] == model_name]
    correct_count = subset["is_correct"].sum()
    total_count = len(subset)
    rate = (correct_count / total_count * 100) if total_count > 0 else 0
    avg_cost = subset["cost_usd"].mean()
    print(f"  {model_name:<15s} ({MODELS[model_name]['label']:>12s}):  {correct_count}/{total_count} correct ({rate:.0f}%)  avg ${avg_cost:.4f}")

# Accuracy by investment
print("\n" + "=" * 75)
print("ACCURACY RATE BY INVESTMENT LEVEL (across all models and problems)")
print("=" * 75)
for inv in INVESTMENT_CONSTRAINTS:
    subset = df[df["investment"] == inv]
    correct_count = subset["is_correct"].sum()
    total_count = len(subset)
    rate = (correct_count / total_count * 100) if total_count > 0 else 0
    avg_cost = subset["cost_usd"].mean()
    avg_chars = subset["output_chars"].mean()
    print(f"  {inv:<12s}:  {correct_count}/{total_count} correct ({rate:.0f}%)  avg ${avg_cost:.4f}  avg {avg_chars:.0f} chars")

CORRECTNESS: Did each model get the right answer?

--- Simple math (avg_speed) --- correct: 71.11 km/h
  Model           Investment       Answer   Correct?    Error       Cost
  -----------------------------------------------------------------
  gpt-4o-mini     quick             71.11        YES     0.0% $   0.0007
  gpt-4o-mini     standard          71.11        YES     0.0% $   0.0007
  gpt-4o-mini     thorough          71.11        YES     0.0% $   0.0009
  gpt-4o          quick             71.11        YES     0.0% $   0.0109
  gpt-4o          standard          71.11        YES     0.0% $   0.0107
  gpt-4o          thorough          71.11        YES     0.0% $   0.0134
  gpt-5.2         quick            640.00         NO   800.0% $   0.0152
  gpt-5.2         standard          71.11        YES     0.0% $   0.0198
  gpt-5.2         thorough         320.00         NO   350.0% $   0.0302

--- Multi-step logic (discount) --- correct: 32.0 %
  Model           Investment       Answer   Co

## 6. Step Completeness

In [9]:
print("=" * 75)
print("STEP COMPLETENESS: Did the model follow all 5 reasoning steps?")
print("=" * 75)
print(f"Steps: {' | '.join(STEPS)}")

for model_name in MODELS:
    subset = [r for r in all_results if r["model"] == model_name]
    print(f"\n  {model_name} ({MODELS[model_name]['label']}):")
    for r in subset:
        steps_str = "".join(
            "Y" if r["steps_detail"].get(s, False) else "-"
            for s in STEPS
        )
        print(
            f"    {r['problem_id']:<12s} {r['investment']:<10s}  "
            f"[{steps_str}]  {r['steps_found']}/5"
        )

# Average steps by model
print("\n  Summary:")
for model_name in MODELS:
    avg = df[df["model"] == model_name]["steps_found"].mean()
    print(f"    {model_name:<15s}: {avg:.1f}/5 avg steps")

STEP COMPLETENESS: Did the model follow all 5 reasoning steps?
Steps: UNDERSTAND | PLAN | EXECUTE | VERIFY | CONCLUDE

  gpt-4o-mini (lightweight):
    avg_speed    quick       [YYYYY]  5/5
    avg_speed    standard    [YYYYY]  5/5
    avg_speed    thorough    [YYYYY]  5/5
    discount     quick       [YYYYY]  5/5
    discount     standard    [YYYYY]  5/5
    discount     thorough    [YYYYY]  5/5
    uptime       quick       [YYYYY]  5/5
    uptime       standard    [YYYYY]  5/5
    uptime       thorough    [YYYYY]  5/5

  gpt-4o (mid-tier):
    avg_speed    quick       [YYYYY]  5/5
    avg_speed    standard    [YYYYY]  5/5
    avg_speed    thorough    [YYYYY]  5/5
    discount     quick       [YYYYY]  5/5
    discount     standard    [YYYYY]  5/5
    discount     thorough    [YYYYY]  5/5
    uptime       quick       [YYYYY]  5/5
    uptime       standard    [YYYYY]  5/5
    uptime       thorough    [YYYYY]  5/5

  gpt-5.2 (reasoning):
    avg_speed    quick       [YYYYY]  5/5
    avg_

## 7. Quality Judge: LLM-as-Judge Scoring

In [10]:
JUDGE_PROMPT = """You are an expert evaluator of mathematical/logical reasoning. You will receive:
- The PROBLEM that was solved
- The CORRECT ANSWER
- The MODEL'S OUTPUT (step-by-step solution)

Score the OUTPUT on four dimensions (1-10 scale, 10 = best):

1. **Correctness** (1-10): Is the final answer correct? Are intermediate calculations accurate?
2. **Reasoning Quality** (1-10): Are the logical steps sound? Is each step justified?
3. **Clarity** (1-10): Is the solution easy to follow? Good formatting and explanation?
4. **Completeness** (1-10): Does it show all work? Verify the answer? Provide confidence assessment?

Return ONLY valid JSON (no markdown fences):
{"correctness": <int>, "reasoning_quality": <int>, "clarity": <int>, "completeness": <int>, "reasoning": "<1-2 sentence justification>"}
"""


def judge_reasoning(output_text: str, problem: dict) -> dict:
    """Score a single reasoning output using LLM-as-judge."""
    prompt = (
        f"{JUDGE_PROMPT}\n\n"
        f"---\nPROBLEM: {problem['problem']}\n\n"
        f"CORRECT ANSWER: {problem['correct_answer']} {problem['unit']}\n\n"
        f"MODEL OUTPUT:\n{output_text[:4000]}\n---"
    )
    ctx = Context(directive=Directive(content=prompt))
    result = ctx.execute(provider="openai", model="gpt-4o", max_tokens=300, use_cache=False)
    raw = (result.response or "").strip()
    if raw.startswith("```"):
        lines = raw.split("\n")
        raw = "\n".join(lines[1:-1] if len(lines) > 2 else lines)
        raw = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"correctness": 0, "reasoning_quality": 0, "clarity": 0, "completeness": 0, "reasoning": f"Parse error: {raw[:200]}"}


print("Judge function ready.")

Judge function ready.


In [11]:
quality_scores = []

for idx, r in enumerate(all_results):
    problem = next(p for p in PROBLEMS if p["id"] == r["problem_id"])
    label = f"{r['model']} x {r['problem_id']} x {r['investment']}"
    print(f"{idx+1}/{len(all_results)}  Judging {label}...", end=" ", flush=True)

    if not r["full_output"]:
        scores = {"correctness": 0, "reasoning_quality": 0, "clarity": 0, "completeness": 0, "reasoning": "No output"}
    else:
        scores = judge_reasoning(r["full_output"], problem)

    scores["model"] = r["model"]
    scores["problem_id"] = r["problem_id"]
    scores["investment"] = r["investment"]
    scores["label"] = label
    quality_scores.append(scores)

    avg = sum(scores.get(k, 0) for k in ["correctness", "reasoning_quality", "clarity", "completeness"]) / 4
    print(f"C={scores.get('correctness')} R={scores.get('reasoning_quality')} Cl={scores.get('clarity')} Co={scores.get('completeness')}  avg={avg:.1f}")

print("\nAll judging complete.")

1/27  Judging gpt-4o-mini x avg_speed x quick... C=10 R=10 Cl=10 Co=10  avg=10.0
2/27  Judging gpt-4o-mini x avg_speed x standard... C=10 R=10 Cl=10 Co=10  avg=10.0
3/27  Judging gpt-4o-mini x avg_speed x thorough... C=10 R=10 Cl=10 Co=10  avg=10.0
4/27  Judging gpt-4o-mini x discount x quick... C=10 R=10 Cl=10 Co=10  avg=10.0
5/27  Judging gpt-4o-mini x discount x standard... C=10 R=10 Cl=10 Co=10  avg=10.0
6/27  Judging gpt-4o-mini x discount x thorough... C=10 R=10 Cl=10 Co=10  avg=10.0
7/27  Judging gpt-4o-mini x uptime x quick... C=10 R=10 Cl=10 Co=10  avg=10.0
8/27  Judging gpt-4o-mini x uptime x standard... C=10 R=10 Cl=10 Co=10  avg=10.0
9/27  Judging gpt-4o-mini x uptime x thorough... C=10 R=10 Cl=10 Co=10  avg=10.0
10/27  Judging gpt-4o x avg_speed x quick... C=10 R=10 Cl=10 Co=10  avg=10.0
11/27  Judging gpt-4o x avg_speed x standard... C=10 R=10 Cl=10 Co=10  avg=10.0
12/27  Judging gpt-4o x avg_speed x thorough... C=10 R=10 Cl=10 Co=10  avg=10.0
13/27  Judging gpt-4o x disc

In [12]:
df_quality = pd.DataFrame(quality_scores)
df_quality["avg_score"] = df_quality[["correctness", "reasoning_quality", "clarity", "completeness"]].mean(axis=1).round(1)

display_q = ["model", "problem_id", "investment", "correctness", "reasoning_quality", "clarity", "completeness", "avg_score", "reasoning"]
df_quality[display_q].style.background_gradient(
    subset=["avg_score"], cmap="RdYlGn", vmin=1, vmax=10
).background_gradient(
    subset=["correctness"], cmap="RdYlGn", vmin=1, vmax=10
)

,model,problem_id,investment,correctness,reasoning_quality,clarity,completeness,avg_score,reasoning
0,gpt-4o-mini,avg_speed,quick,10,10,10,10,10.000000,"The solution correctly calculates the average speed by summing distances and times, then dividing. Each step is logically justified, clearly explained, and verified, ensuring complete and correct reasoning."
1,gpt-4o-mini,avg_speed,standard,10,10,10,10,10.000000,"The solution accurately calculates the average speed using correct formulas and logical steps. Each part of the process is thoroughly explained, ensuring clarity and completeness."
2,gpt-4o-mini,avg_speed,thorough,10,10,10,10,10.000000,The solution correctly calculates the average speed with accurate intermediate steps and clear explanations. It ensures verification and provides a thorough understanding of the problem.
3,gpt-4o-mini,discount,quick,10,10,10,10,10.000000,"The solution correctly calculates the total discount as 32%, using both successive discount formulas and alternative verification, ensuring accuracy and clarity throughout the process."
4,gpt-4o-mini,discount,standard,10,10,10,10,10.000000,"The solution accurately calculates the total discount percentage, clearly explains each step, and verifies the result with logical reasoning, ensuring the solution is complete and easy to follow."
5,gpt-4o-mini,discount,thorough,10,10,10,10,10.000000,"The solution correctly applies sequential percentage discounts and compares them to a single discount, showing all necessary calculations and verifying the result with an example."
6,gpt-4o-mini,uptime,quick,10,10,10,10,10.000000,"The model accurately calculated the overall uptime and downtime, clearly explained each step, and verified the results. The solution is complete and well-reasoned."
7,gpt-4o-mini,uptime,standard,10,10,10,10,10.000000,The solution is mathematically correct and proceeds logically from independent uptime calculations to overall uptime and expected downtime. All steps are clearly explained and verified.
8,gpt-4o-mini,uptime,thorough,10,10,10,10,10.000000,"The solution accurately calculates the overall uptime and expected downtime, using clear logical steps and verifying the answer with multiple approaches."
9,gpt-4o,avg_speed,quick,10,10,10,10,10.000000,"The solution correctly calculates the average speed by accurately determining total distance and time, applies the correct formula, and verifies the result with clear and logical explanations at each step."


## 8. Analysis: Model Comparison

In [13]:
print("=" * 75)
print("MODEL COMPARISON (averaged across all problems and investments)")
print("=" * 75)

for model_name in MODELS:
    m_df = df[df["model"] == model_name]
    m_q = df_quality[df_quality["model"] == model_name]
    correct_rate = m_df["is_correct"].mean() * 100
    avg_steps = m_df["steps_found"].mean()
    avg_tokens = m_df["total_tokens"].mean()
    avg_cost = m_df["cost_usd"].mean()
    total_cost = m_df["cost_usd"].sum()
    avg_quality = m_q["avg_score"].mean()
    avg_correctness_score = m_q["correctness"].mean()
    avg_reasoning_score = m_q["reasoning_quality"].mean()

    reasoning_flag = " (reasoning)" if MODELS[model_name]["reasoning"] else ""
    print(f"\n  {model_name}{reasoning_flag}  [{MODELS[model_name]['label']}]")
    print(f"    Accuracy:          {correct_rate:.0f}% correct")
    print(f"    Quality (judge):   {avg_quality:.1f}/10 avg  (correctness: {avg_correctness_score:.1f}, reasoning: {avg_reasoning_score:.1f})")
    print(f"    Steps:             {avg_steps:.1f}/5 avg")
    print(f"    Tokens:            {avg_tokens:.0f} avg per run")
    print(f"    Cost:              ${avg_cost:.4f} avg  (${total_cost:.4f} total for 9 runs)")

# Head-to-head
print("\n" + "=" * 75)
print("HEAD-TO-HEAD: gpt-4o-mini vs gpt-5.2")
print("=" * 75)
mini = df[df["model"] == "gpt-4o-mini"]
big = df[df["model"] == "gpt-5.2"]
mini_q = df_quality[df_quality["model"] == "gpt-4o-mini"]
big_q = df_quality[df_quality["model"] == "gpt-5.2"]

mini_acc = mini["is_correct"].mean() * 100
big_acc = big["is_correct"].mean() * 100
mini_qual = mini_q["avg_score"].mean()
big_qual = big_q["avg_score"].mean()
mini_cost = mini["cost_usd"].sum()
big_cost = big["cost_usd"].sum()
cost_ratio = big_cost / mini_cost if mini_cost > 0 else 0

print(f"\n  {'Metric':<25s} {'gpt-4o-mini':>15s} {'gpt-5.2':>15s}")
print(f"  {'-'*55}")
print(f"  {'Accuracy':<25s} {mini_acc:>14.0f}% {big_acc:>14.0f}%")
print(f"  {'Quality (avg judge)':<25s} {mini_qual:>14.1f} {big_qual:>14.1f}")
print(f"  {'Total cost (9 runs)':<25s} ${mini_cost:>13.4f} ${big_cost:>13.4f}")
print(f"  {'Cost ratio':<25s} {'1x':>15s} {f'{cost_ratio:.0f}x':>15s}")

MODEL COMPARISON (averaged across all problems and investments)

  gpt-4o-mini  [lightweight]
    Accuracy:          89% correct
    Quality (judge):   10.0/10 avg  (correctness: 10.0, reasoning: 10.0)
    Steps:             5.0/5 avg
    Tokens:            1920 avg per run
    Cost:              $0.0008 avg  ($0.0070 total for 9 runs)

  gpt-4o  [mid-tier]
    Accuracy:          100% correct
    Quality (judge):   9.9/10 avg  (correctness: 9.8, reasoning: 9.8)
    Steps:             5.0/5 avg
    Tokens:            1856 avg per run
    Cost:              $0.0123 avg  ($0.1109 total for 9 runs)

  gpt-5.2 (reasoning)  [reasoning]
    Accuracy:          11% correct
    Quality (judge):   9.7/10 avg  (correctness: 9.9, reasoning: 9.8)
    Steps:             5.0/5 avg
    Tokens:            2554 avg per run
    Cost:              $0.0256 avg  ($0.2302 total for 9 runs)

HEAD-TO-HEAD: gpt-4o-mini vs gpt-5.2

  Metric                        gpt-4o-mini         gpt-5.2
  --------------------

## 9. Analysis: Investment Level Comparison

In [14]:
print("=" * 75)
print("INVESTMENT COMPARISON: Does depth control work without killing accuracy?")
print("=" * 75)

for inv in ["quick", "standard", "thorough"]:
    inv_df = df[df["investment"] == inv]
    inv_q = df_quality[df_quality["investment"] == inv]
    acc = inv_df["is_correct"].mean() * 100
    avg_chars = inv_df["output_chars"].mean()
    avg_tokens = inv_df["total_tokens"].mean()
    avg_cost = inv_df["cost_usd"].mean()
    avg_quality = inv_q["avg_score"].mean()
    avg_steps = inv_df["steps_found"].mean()

    print(f"\n  {inv.upper()}:")
    print(f"    Accuracy:     {acc:.0f}%")
    print(f"    Quality:      {avg_quality:.1f}/10")
    print(f"    Steps:        {avg_steps:.1f}/5")
    print(f"    Avg chars:    {avg_chars:.0f}")
    print(f"    Avg tokens:   {avg_tokens:.0f}")
    print(f"    Avg cost:     ${avg_cost:.4f}")

# Quick vs thorough cost savings
quick_cost = df[df["investment"] == "quick"]["cost_usd"].mean()
thorough_cost = df[df["investment"] == "thorough"]["cost_usd"].mean()
quick_acc = df[df["investment"] == "quick"]["is_correct"].mean() * 100
thorough_acc = df[df["investment"] == "thorough"]["is_correct"].mean() * 100
if thorough_cost > 0:
    savings = (1 - quick_cost / thorough_cost) * 100
    print(f"\n  Quick vs Thorough:")
    print(f"    Cost savings: {savings:.0f}%")
    print(f"    Accuracy:     {quick_acc:.0f}% vs {thorough_acc:.0f}%")

INVESTMENT COMPARISON: Does depth control work without killing accuracy?

  QUICK:
    Accuracy:     67%
    Quality:      9.9/10
    Steps:        5.0/5
    Avg chars:    3738
    Avg tokens:   1823
    Avg cost:     $0.0094

  STANDARD:
    Accuracy:     78%
    Quality:      9.8/10
    Steps:        5.0/5
    Avg chars:    4534
    Avg tokens:   2055
    Avg cost:     $0.0124

  THOROUGH:
    Accuracy:     56%
    Quality:      9.9/10
    Steps:        5.0/5
    Avg chars:    5538
    Avg tokens:   2451
    Avg cost:     $0.0168

  Quick vs Thorough:
    Cost savings: 44%
    Accuracy:     67% vs 56%


## 10. Verdict

In [15]:
print("=" * 75)
print("EXPERIMENT VERDICT")
print("=" * 75)

# Question 1: Does template make cheap models reason well?
mini_acc = df[df["model"] == "gpt-4o-mini"]["is_correct"].mean() * 100
big_acc = df[df["model"] == "gpt-5.2"]["is_correct"].mean() * 100
mini_qual = df_quality[df_quality["model"] == "gpt-4o-mini"]["avg_score"].mean()
big_qual = df_quality[df_quality["model"] == "gpt-5.2"]["avg_score"].mean()
mini_total_cost = df[df["model"] == "gpt-4o-mini"]["cost_usd"].sum()
big_total_cost = df[df["model"] == "gpt-5.2"]["cost_usd"].sum()

mid_acc = df[df["model"] == "gpt-4o"]["is_correct"].mean() * 100

print("\n  Q1: Does the template make lightweight models reason as well as reasoning models?")
print(f"     Accuracy  -> gpt-4o-mini: {mini_acc:.0f}%  |  gpt-4o: {mid_acc:.0f}%  |  gpt-5.2: {big_acc:.0f}%")
print(f"     Quality   -> gpt-4o-mini: {mini_qual:.1f}  |  gpt-5.2: {big_qual:.1f}")
print(f"     Cost      -> gpt-4o-mini: ${mini_total_cost:.4f}  |  gpt-5.2: ${big_total_cost:.4f}")

gap = abs(mini_acc - big_acc)
if gap <= 15:
    print(f"  -> YES: Models are within {gap:.0f}pp — template largely bridges the gap.")
    if mini_total_cost < big_total_cost:
        ratio = big_total_cost / mini_total_cost if mini_total_cost > 0 else 0
        print(f"     gpt-4o-mini achieves this at {ratio:.0f}x lower cost.")
elif mini_acc > big_acc:
    print(f"  -> SURPRISING: gpt-4o-mini ({mini_acc:.0f}%) BEATS gpt-5.2 ({big_acc:.0f}%)!")
    print(f"     The template + lightweight model outperforms the reasoning model.")
    print(f"     NOTE: If quality scores disagree, answer-extraction may be the issue.")
else:
    print(f"  -> NO: gpt-5.2 ({big_acc:.0f}%) outperforms gpt-4o-mini ({mini_acc:.0f}%).")
    print(f"     Reasoning models add real value for reasoning tasks.")

# Question 2: Do investment levels work?
quick_acc = df[df["investment"] == "quick"]["is_correct"].mean() * 100
standard_acc = df[df["investment"] == "standard"]["is_correct"].mean() * 100
thorough_acc = df[df["investment"] == "thorough"]["is_correct"].mean() * 100
quick_chars = df[df["investment"] == "quick"]["output_chars"].mean()
thorough_chars = df[df["investment"] == "thorough"]["output_chars"].mean()

print("\n  Q2: Do investment levels control depth without degrading correctness?")
depth_works = thorough_chars > quick_chars * 1.2
accuracy_stable = abs(quick_acc - thorough_acc) <= 20
if depth_works and accuracy_stable:
    print(f"  -> YES: Depth scales ({quick_chars:.0f} -> {thorough_chars:.0f} chars) while accuracy holds")
    print(f"     Quick: {quick_acc:.0f}%  Standard: {standard_acc:.0f}%  Thorough: {thorough_acc:.0f}%")
elif depth_works:
    print(f"  -> PARTIALLY: Depth scales but accuracy varies significantly")
    print(f"     Quick: {quick_acc:.0f}%  Standard: {standard_acc:.0f}%  Thorough: {thorough_acc:.0f}%")
else:
    print(f"  -> NO: Investment constraints did not meaningfully change output depth")

# Question 3: The big one
mini_thorough = df[(df["model"] == "gpt-4o-mini") & (df["investment"] == "thorough")]
big_standard = df[(df["model"] == "gpt-5.2") & (df["investment"] == "standard")]
mini_t_acc = mini_thorough["is_correct"].mean() * 100
big_s_acc = big_standard["is_correct"].mean() * 100
mini_t_cost = mini_thorough["cost_usd"].mean()
big_s_cost = big_standard["cost_usd"].mean()

print("\n  Q3: Does gpt-4o-mini + thorough match gpt-5.2 + standard?")
print(f"     gpt-4o-mini thorough: {mini_t_acc:.0f}% correct, avg ${mini_t_cost:.4f}")
print(f"     gpt-5.2 standard:     {big_s_acc:.0f}% correct, avg ${big_s_cost:.4f}")
acc_gap = abs(mini_t_acc - big_s_acc)
if acc_gap <= 15:
    savings = (1 - mini_t_cost / big_s_cost) * 100 if big_s_cost > 0 else 0
    print(f"  -> YES: Within {acc_gap:.0f}pp accuracy gap, {savings:.0f}% lower cost")
elif mini_t_acc > big_s_acc:
    savings = (1 - mini_t_cost / big_s_cost) * 100 if big_s_cost > 0 else 0
    print(f"  -> YES (mini wins): gpt-4o-mini+thorough beats gpt-5.2+standard by {mini_t_acc - big_s_acc:.0f}pp at {savings:.0f}% lower cost")
else:
    print(f"  -> NO: gpt-5.2+standard leads by {big_s_acc - mini_t_acc:.0f}pp — reasoning model has a meaningful advantage")

EXPERIMENT VERDICT

  Q1: Does the template make lightweight models reason as well as reasoning models?
  -> NO: gpt-5.2 (11%) significantly outperforms gpt-4o-mini (89%)
     Reasoning models add real value for reasoning tasks.
     Quality: 10.0 vs 9.7

  Q2: Do investment levels control depth without degrading correctness?
  -> YES: Depth scales (3738 -> 5538 chars) while accuracy holds
     Quick: 67%  Standard: 78%  Thorough: 56%

  Q3: Does gpt-4o-mini + thorough = gpt-5.2 + standard?
     gpt-4o-mini thorough: 67% correct, avg $0.0009
     gpt-5.2 standard:     33% correct, avg $0.0244
  -> YES: Comparable accuracy at 97% lower cost
